## Temperature Time-Line Data

### Import Libraries

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.optimizers import Adam

### Load Dataset

In [2]:
df = pd.read_csv('data.csv', parse_dates=['Date'], index_col='Date')
print(df.head())

            Temperature
Date                   
2010-01-01    27.483571
2010-01-02    24.308678
2010-01-03    28.238443
2010-01-04    32.615149
2010-01-05    23.829233


### Scaling

In [3]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df.values)

### Slicing Dataset

In [4]:
def create_dataset(data, time_step=1):
    X, y = [], []
    for i in range(len(data) - time_step - 1):
        X.append(data[i:(i + time_step), 0])
        y.append(data[i + time_step, 0])
    return np.array(X), np.array(y)

time_step = 100
X, y = create_dataset(scaled_data, time_step)
X = X.reshape(X.shape[0], X.shape[1], 1)

### GRU Model

In [5]:
model = Sequential()
model.add(GRU(units=50, return_sequences=True, input_shape=(X.shape[1], 1)))
model.add(GRU(units=50))
model.add(Dense(units=1))
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [6]:
model.fit(X, y, epochs=10, batch_size=32)

Epoch 1/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 35s 109ms/step - loss: 0.0364
Epoch 2/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 27s 109ms/step - loss: 0.0188
Epoch 3/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 27s 110ms/step - loss: 0.0180
Epoch 4/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 27s 110ms/step - loss: 0.0179
Epoch 5/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 27s 110ms/step - loss: 0.0176
Epoch 6/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 26s 107ms/step - loss: 0.0178
Epoch 7/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 41s 108ms/step - loss: 0.0176
Epoch 8/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 26s 106ms/step - loss: 0.0177
Epoch 9/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 26s 106ms/step - loss: 0.0177
Epoch 10/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 41s 108ms/step - loss: 0.0179


### Prediction

In [18]:
input_sequence = scaled_data[-time_step:].reshape(1, time_step, 1)
predicted_values = model.predict(input_sequence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step


In [19]:
predicted_values = scaler.inverse_transform(predicted_values)
print(f"The predicted temperature for the next day is: {predicted_values[0][0]:.2f}°C")

The predicted temperature for the next day is: 25.49°C


### Saving model

In [20]:
model.save('Temperature_trend_GRU.h5')